# Web App Dev

In [1]:
import base64
import sys
import json
import zlib
from IPython.display import SVG, Image
import requests

mermaid_code = '''
sequenceDiagram
    participant U as 👤 User
    participant UI as 🎨 Gradio Interface
    participant AUTH as 🔐 API Key Manager
    participant PROC as ⚙️ Request Processor
    participant CLAUDE as 🤖 Anthropic Claude
    participant CLEAN as 🧹 Code Cleaner
    participant PREV as 👀 Preview Engine
    participant HIST as 💾 History Manager
    participant DOWN as 📥 Download Handler

    Note over U, DOWN: 🚀 Claude HTML Generator Architecture Flow

    U->>UI: 1. Enter website description
    U->>UI: 2. Provide Anthropic API key
    U->>UI: 3. Select Claude model (Opus/Sonnet/Haiku)

    UI->>AUTH: 4. Validate API key format
    AUTH-->>UI: ✅ Key validated

    UI->>PROC: 5. Process user input & prepare request

    Note over PROC: 📝 Formats system prompt<br/>& conversation history

    PROC->>CLAUDE: 6. Send prompt to Claude API
    Note over CLAUDE: 🧠 Claude generates<br/>HTML/CSS/JS code

    CLAUDE-->>PROC: 7. Return generated code

    PROC->>CLEAN: 8. Clean markdown blocks
    Note over CLEAN: 🔧 Remove ```html``` blocks<br/>& extract pure HTML

    CLEAN-->>UI: 9. Return cleaned HTML code

    par Display Code
        UI->>UI: 10a. Show code in editor tab
    and Generate Preview
        UI->>PREV: 10b. Create safe preview
        Note over PREV: 🔒 Wrap in iframe with<br/>sandbox security
        PREV-->>UI: 📺 Return iframe preview
    and Update History
        UI->>HIST: 10c. Update conversation
        HIST-->>UI: 💬 Updated chat history
    end

    UI-->>U: 11. Display all results

    alt User wants to download
        U->>UI: 12. Click download button
        UI->>DOWN: 13. Prepare HTML file
        DOWN-->>U: 📁 Download complete HTML file
    end

    alt User wants to iterate
        U->>UI: 14. Enter modification request
        Note over UI, CLAUDE: 🔄 Process repeats with<br/>conversation context
    end

    Note over U, DOWN: ✨ Complete HTML website generated!
'''

def js_btoa(data):
    return base64.b64encode(data)

def pako_deflate(data):
    compress = zlib.compressobj(9, zlib.DEFLATED, 15, 8, zlib.Z_DEFAULT_STRATEGY)
    compressed_data = compress.compress(data)
    compressed_data += compress.flush()
    return compressed_data

def genPakoLink(graphMarkdown: str):
    jGraph = {"code": graphMarkdown, "mermaid": {"theme": "standard"}}
    byteStr = json.dumps(jGraph).encode('utf-8')
    deflated = pako_deflate(byteStr)
    dEncode = js_btoa(deflated)
    link_code = dEncode.decode('ascii')
    return link_code

mermaid_link = genPakoLink(mermaid_code)
print("mermaid.live link:")
print('http://mermaid.live/edit#pako:' + mermaid_link)

mermaid.live link:
http://mermaid.live/edit#pako:eNp9Vl1vGjsQ/StzeajulRIghCQEVZEQpIXeNqlKaF94iLFng8Wuvdf2hqKq//2Od9f7EVB5ACXMmTkzc+YsvzpcC+yMobNWFv/LUHGcSfZiWLJWQK+UGSe5TJlysAJmYZ2J0aWgd349hJVFcyJuUQVyeo/YCD4aJqSGhXJoIsbxGDRZPc1b+cVFHyZfF/AvHuALU+zlVKmv3x6nBWpwfXu7ziLsR/DNN2LpS6M5WqtP4KafJ6vZfVUP83rXMFFua3QqOUxjlgk8BbyfPLRx0S1MaYgEQaZOk7z/3h7dsE/k8FXiHu7Vi1QnCs0Xy6c2aIMwl9Zp84d5zB5/PLRReAUzvVexZgLmTInYwwrgg3YI+hUNrM5y5LjG4ahfzgDmT18+w0ek1hgVh4nhW+mQu8wgfIj1PqRbnd/drRZjuOhSU7Rn2OPGUiQItNzI1En9JnLQ9Tt6lVSknrzf+Q4P7cjLLiwxpqKBVEITj+HvxzSzvaVWCl1vzuQu+6disyCoF9UYhl34zmIpGHEps0OkTcJcEeqjzstCJKSb/lUuutcSI1opveLGcNUN6oKMbgCkSjMH7yA1SOtAMIUEjyddwOv1CAEfcioW7ME6TCiFTlL3fmN6d++Aa0Uoy/zsYFtsPyT1qYhQIeUxXPsRKVHiwekwKmr5LYuAqUWM1bpfik2jzSn45femy2Xv0xK8VYTiRYbzaiA3XTo7koSq8KIVX5Gl8xnDqFtcCyTM7ATJEzax5jt7zDMPrz1hcENlEvoOnp+fty6J6aPElhPDn84w0knq5enZ14wpV1jzbUWX51crCpU3GdMeYSZtGrNDft/Ffysd5ErvMxr6Vu9zIIkAUEh/I45tinA6uHA6GE7+TSLvDj7VhmZi0MdZFqFXUju4qSGPaBjlAH4Ylvr6MiLnRthLt83nYYnARv8Eizwz0h3qdD5JLfogx4i

## Install

In [2]:
pip install gradio openai

## Build a Web App Dev Assistant (OpenAI)

In [3]:
import os
import re
import base64
import gradio as gr
from openai import OpenAI

# Configuration
SYSTEM_PROMPT = """You are an expert web developer. Create modern, responsive HTML websites using only HTML, CSS, and JavaScript.

REQUIREMENTS:
- Use modern CSS for styling (flexbox, grid, modern selectors)
- Make it fully responsive for mobile devices
- Include proper semantic HTML structure
- Use modern JavaScript ES6+ features when needed
- Create clean, professional designs with good UX
- Include proper accessibility features

IMPORTANT:
- Always respond with complete, working HTML code
- Include all CSS in <style> tags within the HTML
- Include all JavaScript in <script> tags within the HTML
- Make it a single, self-contained HTML file
- Do NOT include any explanations - only return the HTML code
- Do NOT wrap the code in markdown code blocks

The HTML should be ready to run directly in a browser."""

# OpenAI Client
def get_openai_client(api_key):
    """Initialize OpenAI client with provided API key"""
    if not api_key:
        return None
    return OpenAI(api_key=api_key)

def remove_code_block(text):
    """Remove markdown code blocks if present"""
    # Try to match code blocks with language markers
    patterns = [
        r'```(?:html|HTML)\n([\s\S]+?)\n```',  # Match ```html or ```HTML
        r'```\n([\s\S]+?)\n```',               # Match code blocks without language markers
        r'```([\s\S]+?)```'                      # Match code blocks without line breaks
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            extracted = match.group(1).strip()
            return extracted

    # If no code block is found, check if the entire text is HTML
    if text.strip().startswith('<!DOCTYPE html>') or text.strip().startswith('<html'):
        return text.strip()

    return text.strip()

def send_to_preview(html_code):
    """Create a safe preview of the HTML code"""
    if not html_code or not html_code.strip():
        return "<div style='padding:2em;color:#666;text-align:center;'>No HTML code to preview</div>"

    # Add a wrapper to ensure proper rendering
    wrapped_code = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <script>
            // Safe localStorage polyfill for iframe
            const safeStorage = {{
                _data: {{}},
                getItem: function(key) {{ return this._data[key] || null; }},
                setItem: function(key, value) {{ this._data[key] = value; }},
                removeItem: function(key) {{ delete this._data[key]; }},
                clear: function() {{ this._data = {{}}; }}
            }};

            if (!window.localStorage) {{
                Object.defineProperty(window, 'localStorage', {{
                    value: safeStorage,
                    writable: false
                }});
            }}

            window.onerror = function(message, source, lineno, colno, error) {{
                console.error('Error:', message);
            }};
        </script>
    </head>
    <body>
        {html_code}
    </body>
    </html>
    """

    # Encode HTML for safe iframe embedding
    encoded_html = base64.b64encode(wrapped_code.encode('utf-8')).decode('utf-8')
    data_uri = f"data:text/html;charset=utf-8;base64,{encoded_html}"
    iframe = f'<iframe src="{data_uri}" width="100%" height="600px" style="border:1px solid #ddd;border-radius:8px;" sandbox="allow-scripts allow-same-origin allow-forms allow-popups allow-modals"></iframe>'
    return iframe

def generate_html_code(prompt, api_key, history):
    """Generate HTML code using OpenAI API"""
    if not prompt.strip():
        return "Please enter a description of what you want to build.", history, ""

    if not api_key:
        return "Please enter your OpenAI API key.", history, ""

    try:
        client = get_openai_client(api_key)
        if not client:
            return "Failed to initialize OpenAI client.", history, ""

        # Prepare messages for the API
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]

        # Add conversation history
        for user_msg, assistant_msg in history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": assistant_msg})

        # Add current user message
        messages.append({"role": "user", "content": prompt})

        # Call OpenAI API
        response = client.chat.completions.create(
            model="gpt-4",  # You can change this to gpt-3.5-turbo for lower cost
            messages=messages,
            max_tokens=4000,
            temperature=0.7
        )

        # Extract the generated code
        generated_code = response.choices[0].message.content
        clean_code = remove_code_block(generated_code)

        # Update history
        new_history = history + [(prompt, clean_code)]

        # Generate preview
        preview_html = send_to_preview(clean_code)

        return clean_code, new_history, preview_html

    except Exception as e:
        error_msg = f"Error generating code: {str(e)}"
        return error_msg, history, ""

def clear_all():
    """Clear all inputs and outputs"""
    return "", [], "", ""

# Demo examples
DEMO_EXAMPLES = [
    "Build a modern landing page for a tech startup",
    "Create a responsive portfolio website for a photographer",
    "Design a dashboard for a fitness tracking app",
    "Build a restaurant menu website with modern styling",
    "Create a blog layout with sidebar and article cards",
    "Design a contact form with validation",
    "Build a pricing page with three tier options",
    "Create an image gallery with lightbox effect"
]

# Main Gradio Interface
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="gray",
        font=gr.themes.GoogleFont("Inter")
    ),
    title="OpenAI HTML Generator"
) as demo:

    # State for conversation history
    history_state = gr.State([])

    gr.HTML("""
    <div style="text-align: center; margin-bottom: 2rem;">
        <h1 style="color: #333; margin-bottom: 0.5rem;">🎨 OpenAI HTML Generator</h1>
        <p style="color: #666; font-size: 1.1rem;">Describe your website and watch it come to life!</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group():
                gr.Markdown("### ⚙️ Configuration")
                api_key_input = gr.Textbox(
                    label="OpenAI API Key",
                    placeholder="sk-...",
                    type="password",
                    info="Your OpenAI API key (kept secure in your session)"
                )

            with gr.Group():
                gr.Markdown("### 💬 What do you want to build?")
                prompt_input = gr.Textbox(
                    label="Describe your website",
                    placeholder="e.g., Build a modern landing page for a tech startup with hero section, features, and contact form",
                    lines=3
                )

                with gr.Row():
                    generate_btn = gr.Button("🚀 Generate HTML", variant="primary", scale=2)
                    clear_btn = gr.Button("🗑️ Clear", variant="secondary", scale=1)

            with gr.Group():
                gr.Markdown("### 🎯 Quick Examples")
                with gr.Column():
                    example_buttons = []
                    for i, example in enumerate(DEMO_EXAMPLES[:4]):
                        btn = gr.Button(example, variant="secondary", size="sm")
                        example_buttons.append(btn)
                        btn.click(
                            fn=lambda x=example: x,
                            outputs=prompt_input
                        )

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("📝 Generated Code"):
                    code_output = gr.Code(
                        label="HTML Code",
                        language="html",
                        lines=20,
                        interactive=True,
                        show_label=False
                    )

                    with gr.Row():
                        download_btn = gr.DownloadButton(
                            "💾 Download HTML",
                            variant="secondary"
                        )

                with gr.Tab("👀 Live Preview"):
                    preview_output = gr.HTML(
                        label="Preview",
                        value="<div style='padding:2em;color:#666;text-align:center;'>Generate some HTML code to see the preview here!</div>"
                    )

                with gr.Tab("💬 Chat History"):
                    chat_output = gr.Chatbot(
                        label="Conversation History",
                        height=400,
                        show_label=False
                    )

    # Event handlers
    def on_generate(prompt, api_key, history):
        code, new_history, preview = generate_html_code(prompt, api_key, history)

        # Format history for chatbot display
        chat_history = []
        for user_msg, assistant_msg in new_history:
            chat_history.append([user_msg, f"Generated HTML code with {len(assistant_msg)} characters"])

        return code, new_history, preview, chat_history, ""

    def on_clear():
        return "", [], "", [], "<div style='padding:2em;color:#666;text-align:center;'>Generate some HTML code to see the preview here!</div>"

    def update_preview(code):
        if code.strip():
            return send_to_preview(code)
        return "<div style='padding:2em;color:#666;text-align:center;'>No HTML code to preview</div>"

    def prepare_download(code):
        if code.strip():
            return gr.DownloadButton(
                "💾 Download HTML",
                value=code,
                filename="generated_website.html",
                variant="secondary"
            )
        return gr.DownloadButton("💾 Download HTML", variant="secondary")

    # Wire up the events
    generate_btn.click(
        fn=on_generate,
        inputs=[prompt_input, api_key_input, history_state],
        outputs=[code_output, history_state, preview_output, chat_output, prompt_input]
    )

    clear_btn.click(
        fn=on_clear,
        outputs=[prompt_input, history_state, code_output, chat_output, preview_output]
    )

    # Update preview when code changes
    code_output.change(
        fn=update_preview,
        inputs=code_output,
        outputs=preview_output
    )

    # Update download button when code changes
    code_output.change(
        fn=prepare_download,
        inputs=code_output,
        outputs=download_btn
    )

    gr.HTML("""
    <div style="text-align: center; margin-top: 2rem; padding: 1rem; border-top: 1px solid #eee;">
        <p style="color: #666; font-size: 0.9rem;">
            🤖 Powered by OpenAI GPT-4 | 🎨 Built with Gradio
        </p>
    </div>
    """)

# Launch the app
if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=False,
        inbrowser=True
    )

/tmp/ipython-input-591877391.py:245: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chat_output = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

## Build a Web App Dev Assistant (Anthropic)

In [4]:
pip install anthropic gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.1/293.1 kB 5.1 MB/s eta 0:00:00


In [11]:
import os
import re
import base64
import gradio as gr
from anthropic import Anthropic

# Configuration
SYSTEM_PROMPT = """You are an expert web developer. Create modern, responsive HTML websites using only HTML, CSS, and JavaScript.

REQUIREMENTS:
- Use modern CSS for styling (flexbox, grid, modern selectors)
- Make it fully responsive for mobile devices
- Include proper semantic HTML structure
- Use modern JavaScript ES6+ features when needed
- Create clean, professional designs with good UX
- Include proper accessibility features

IMPORTANT:
- Always respond with complete, working HTML code
- Include all CSS in <style> tags within the HTML
- Include all JavaScript in <script> tags within the HTML
- Make it a single, self-contained HTML file
- Do NOT include any explanations - only return the HTML code
- Do NOT wrap the code in markdown code blocks

The HTML should be ready to run directly in a browser."""

# Anthropic Client
def get_anthropic_client(api_key):
    """Initialize Anthropic client with provided API key"""
    if not api_key:
        return None
    return Anthropic(api_key=api_key)

def remove_code_block(text):
    """Remove markdown code blocks if present"""
    # Try to match code blocks with language markers
    patterns = [
        r'```(?:html|HTML)\n([\s\S]+?)\n```',  # Match ```html or ```HTML
        r'```\n([\s\S]+?)\n```',               # Match code blocks without language markers
        r'```([\s\S]+?)```'                      # Match code blocks without line breaks
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            extracted = match.group(1).strip()
            return extracted

    # If no code block is found, check if the entire text is HTML
    if text.strip().startswith('<!DOCTYPE html>') or text.strip().startswith('<html'):
        return text.strip()

    return text.strip()

def send_to_preview(html_code):
    """Create a safe preview of the HTML code"""
    if not html_code or not html_code.strip():
        return "<div style='padding:2em;color:#666;text-align:center;'>No HTML code to preview</div>"

    # Add a wrapper to ensure proper rendering
    wrapped_code = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <script>
            // Safe localStorage polyfill for iframe
            const safeStorage = {{
                _data: {{}},
                getItem: function(key) {{ return this._data[key] || null; }},
                setItem: function(key, value) {{ this._data[key] = value; }},
                removeItem: function(key) {{ delete this._data[key]; }},
                clear: function() {{ this._data = {{}}; }}
            }};

            if (!window.localStorage) {{
                Object.defineProperty(window, 'localStorage', {{
                    value: safeStorage,
                    writable: false
                }});
            }}

            window.onerror = function(message, source, lineno, colno, error) {{
                console.error('Error:', message);
            }};
        </script>
    </head>
    <body>
        {html_code}
    </body>
    </html>
    """

    # Encode HTML for safe iframe embedding
    encoded_html = base64.b64encode(wrapped_code.encode('utf-8')).decode('utf-8')
    data_uri = f"data:text/html;charset=utf-8;base64,{encoded_html}"
    iframe = f'<iframe src="{data_uri}" width="100%" height="600px" style="border:1px solid #ddd;border-radius:8px;" sandbox="allow-scripts allow-same-origin allow-forms allow-popups allow-modals"></iframe>'
    return iframe

def generate_html_code(prompt, api_key, history, model="claude-3-opus-20240229"):
    """Generate HTML code using Anthropic Claude API"""
    if not prompt.strip():
        return "Please enter a description of what you want to build.", history, ""

    if not api_key:
        return "Please enter your Anthropic API key.", history, ""

    try:
        client = get_anthropic_client(api_key)
        if not client:
            return "Failed to initialize Anthropic client.", history, ""

        # Prepare messages for the API
        messages = []

        # Add conversation history
        for user_msg, assistant_msg in history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": assistant_msg})

        # Add current user message
        messages.append({"role": "user", "content": prompt})

        # Call Anthropic API
        response = client.messages.create(
            model=model,
            max_tokens=4000,
            temperature=0.7,
            system=SYSTEM_PROMPT,
            messages=messages
        )

        # Extract the generated code
        generated_code = response.content[0].text
        clean_code = remove_code_block(generated_code)

        # Update history
        new_history = history + [(prompt, clean_code)]

        # Generate preview
        preview_html = send_to_preview(clean_code)

        return clean_code, new_history, preview_html

    except Exception as e:
        error_msg = f"Error generating code: {str(e)}"
        return error_msg, history, ""

def clear_all():
    """Clear all inputs and outputs"""
    return "", [], "", ""

# Available Claude models
CLAUDE_MODELS = [
    {"name": "Claude Opus 4", "id": "claude-3-opus-20240229"},
    {"name": "Claude Sonnet 3.5", "id": "claude-3-5-sonnet-20241022"},
    {"name": "Claude Haiku 3.5", "id": "claude-3-5-haiku-20241022"}
]

# Demo examples
DEMO_EXAMPLES = [
    "Build a modern landing page for a tech startup",
    "Create a responsive portfolio website for a photographer",
    "Design a dashboard for a fitness tracking app",
    "Build a restaurant menu website with modern styling",
    "Create a blog layout with sidebar and article cards",
    "Design a contact form with validation",
    "Build a pricing page with three tier options",
    "Create an image gallery with lightbox effect"
]

# Main Gradio Interface
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="orange",
        secondary_hue="gray",
        font=gr.themes.GoogleFont("Inter")
    ),
    title="Claude HTML Generator"
) as demo:

    # State for conversation history
    history_state = gr.State([])

    gr.HTML("""
    <div style="text-align: center; margin-bottom: 2rem;">
        <h1 style="color: #333; margin-bottom: 0.5rem;">🤖 Claude HTML Generator</h1>
        <p style="color: #666; font-size: 1.1rem;">Describe your website and watch Claude bring it to life!</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group():
                gr.Markdown("### ⚙️ Configuration")
                api_key_input = gr.Textbox(
                    label="Anthropic API Key",
                    placeholder="sk-ant-...",
                    type="password",
                    info="Your Anthropic API key (kept secure in your session)"
                )

                model_dropdown = gr.Dropdown(
                    choices=[model["name"] for model in CLAUDE_MODELS],
                    value="Claude Opus 4",
                    label="Claude Model",
                    info="Choose your preferred Claude model"
                )

            with gr.Group():
                gr.Markdown("### 💬 What do you want to build?")
                prompt_input = gr.Textbox(
                    label="Describe your website",
                    placeholder="e.g., Build a modern landing page for a tech startup with hero section, features, and contact form",
                    lines=3
                )

                with gr.Row():
                    generate_btn = gr.Button("🚀 Generate HTML", variant="primary", scale=2)
                    clear_btn = gr.Button("🗑️ Clear", variant="secondary", scale=1)

            with gr.Group():
                gr.Markdown("### 🎯 Quick Examples")
                with gr.Column():
                    example_buttons = []
                    for i, example in enumerate(DEMO_EXAMPLES[:4]):
                        btn = gr.Button(example, variant="secondary", size="sm")
                        example_buttons.append(btn)
                        btn.click(
                            fn=lambda x=example: x,
                            outputs=prompt_input
                        )

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("📝 Generated Code"):
                    code_output = gr.Code(
                        label="HTML Code",
                        language="html",
                        lines=20,
                        interactive=True,
                        show_label=False
                    )

                    with gr.Row():
                        download_btn = gr.DownloadButton(
                            "💾 Download HTML",
                            variant="secondary"
                        )

                with gr.Tab("👀 Live Preview"):
                    preview_output = gr.HTML(
                        label="Preview",
                        value="<div style='padding:2em;color:#666;text-align:center;'>Generate some HTML code to see the preview here!</div>"
                    )

                with gr.Tab("💬 Chat History"):
                    chat_output = gr.Chatbot(
                        label="Conversation History",
                        height=400,
                        show_label=False
                    )

    # Event handlers
    def on_generate(prompt, api_key, model_name, history):
        # Map model name to model ID
        model_id = next((model["id"] for model in CLAUDE_MODELS if model["name"] == model_name), "claude-3-opus-20240229")

        code, new_history, preview = generate_html_code(prompt, api_key, history, model_id)

        # Format history for chatbot display
        chat_history = []
        for user_msg, assistant_msg in new_history:
            chat_history.append([user_msg, f"Generated HTML code with {len(assistant_msg)} characters"])

        return code, new_history, preview, chat_history, ""

    def on_clear():
        return "", [], "", [], "<div style='padding:2em;color:#666;text-align:center;'>Generate some HTML code to see the preview here!</div>"

    def update_preview(code):
        if code.strip():
            return send_to_preview(code)
        return "<div style='padding:2em;color:#666;text-align:center;'>No HTML code to preview</div>"

    def prepare_download(code):
        if code.strip():
            return gr.DownloadButton(
                "💾 Download HTML",
                value=code,
                filename="generated_website.html",
                variant="secondary"
            )
        return gr.DownloadButton("💾 Download HTML", variant="secondary")

    # Wire up the events
    generate_btn.click(
        fn=on_generate,
        inputs=[prompt_input, api_key_input, model_dropdown, history_state],
        outputs=[code_output, history_state, preview_output, chat_output, prompt_input]
    )

    clear_btn.click(
        fn=on_clear,
        outputs=[prompt_input, history_state, code_output, chat_output, preview_output]
    )

    # Update preview when code changes
    code_output.change(
        fn=update_preview,
        inputs=code_output,
        outputs=preview_output
    )

    # Update download button when code changes
    code_output.change(
        fn=prepare_download,
        inputs=code_output,
        outputs=download_btn
    )

    gr.HTML("""
    <div style="text-align: center; margin-top: 2rem; padding: 1rem; border-top: 1px solid #eee;">
        <p style="color: #666; font-size: 0.9rem;">
            🤖 Powered by Anthropic Claude | 🎨 Built with Gradio
        </p>
    </div>
    """)

# Launch the app
if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=7865,
        share=False,
        inbrowser=True
    )

/tmp/ipython-input-2474546211.py:260: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chat_output = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

## Build a Web App Dev Assistant with Voice Input (Anthropic)

In [7]:
pip install SpeechRecognition pydub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 57.7 MB/s eta 0:00:00


Diagram:

In [8]:
import base64
import sys
import json
import zlib
from IPython.display import SVG, Image
import requests

mermaid_code = '''
sequenceDiagram
    participant U as 👤 User
    participant UI as 🎨 Gradio Interface
    participant TAB as 📑 Input Tabs
    participant VOICE as 🎤 Voice Processor
    participant SR as 🗣️ Speech Recognition
    participant AUTH as 🔐 API Key Manager
    participant MERGE as 🔄 Input Merger
    participant PROC as ⚙️ Request Processor
    participant CLAUDE as 🤖 Anthropic Claude
    participant CLEAN as 🧹 Code Cleaner
    participant PREV as 👀 Preview Engine
    participant HIST as 💾 History Manager
    participant DOWN as 📥 Download Handler

    Note over U, DOWN: 🎤🤖 Claude HTML Generator with Voice Input

    alt Text Input Path
        U->>TAB: 1a. Select "Type Description" tab
        U->>UI: 2a. Enter website description (text)
        UI->>MERGE: 3a. Pass text input
    else Voice Input Path
        U->>TAB: 1b. Select "Voice Description" tab
        U->>UI: 2b. Record voice description
        UI->>VOICE: 3b. Process audio file

        Note over VOICE: 🎵 Convert audio format<br/>(AudioSegment processing)

        VOICE->>SR: 4b. Send audio to Speech Recognition
        Note over SR: 🗣️ Google Speech API<br/>converts speech to text

        SR-->>VOICE: 5b. Return transcribed text
        VOICE-->>UI: 6b. Display transcription feedback
        VOICE->>MERGE: 7b. Pass transcribed text
    else Hybrid Input Path
        Note over U, MERGE: 🔄 User can combine both text AND voice
        UI->>MERGE: Both text + voice inputs
    end

    U->>UI: 8. Provide Anthropic API key
    U->>UI: 9. Select Claude model (Opus/Sonnet/Haiku)
    U->>UI: 10. Click "Generate HTML"

    UI->>AUTH: 11. Validate API key format
    AUTH-->>UI: ✅ Key validated

    MERGE->>PROC: 12. Combine all inputs into final prompt
    Note over MERGE, PROC: 📝 Merges text + voice inputs<br/>Creates unified prompt

    PROC->>CLAUDE: 13. Send enhanced prompt to Claude API
    Note over CLAUDE: 🧠 Claude Opus 4 generates<br/>complete HTML/CSS/JS code

    CLAUDE-->>PROC: 14. Return generated code

    PROC->>CLEAN: 15. Clean markdown blocks
    Note over CLEAN: 🔧 Remove ```html``` blocks<br/>Extract pure HTML code

    CLEAN-->>UI: 16. Return cleaned HTML code

    par Display Results
        UI->>UI: 17a. Show code in editor
    and Generate Preview
        UI->>PREV: 17b. Create secure preview
        Note over PREV: 🔒 Iframe with sandbox<br/>Base64 encoding for security
        PREV-->>UI: 📺 Return live preview
    and Update History
        UI->>HIST: 17c. Update conversation
        Note over HIST: 💬 Store user prompt +<br/>generated code for context
        HIST-->>UI: Updated chat history
    and Voice Feedback
        UI->>UI: 17d. Display voice transcription
        Note over UI: 🎤 Show "Voice input transcribed:<br/>'user's spoken words'"
    end

    UI-->>U: 18. Display all results + voice feedback

    alt User wants to download
        U->>UI: 19. Click download button
        UI->>DOWN: 20. Prepare HTML file
        DOWN-->>U: 📁 Download complete website
    end

    alt User wants to iterate (Voice/Text)
        U->>UI: 21. Record new voice input OR type changes
        Note over UI, CLAUDE: 🔄 Process repeats with<br/>conversation context +<br/>new voice/text input
    end

    alt Voice Input Error Handling
        SR-->>VOICE: ❌ "Could not understand audio"
        VOICE-->>UI: 🚨 Display error message
        UI-->>U: "Please try speaking more clearly"
    end

    Note over U, DOWN: ✨ Complete HTML website generated from voice or text!

    rect rgb(255, 248, 220)
        Note over VOICE, SR: 🎤 NEW VOICE FEATURES:<br/>• Real-time transcription<br/>• Audio format conversion<br/>• Google Speech Recognition<br/>• Error handling & feedback<br/>• Hybrid text+voice input
    end
'''

def js_btoa(data):
    return base64.b64encode(data)

def pako_deflate(data):
    compress = zlib.compressobj(9, zlib.DEFLATED, 15, 8, zlib.Z_DEFAULT_STRATEGY)
    compressed_data = compress.compress(data)
    compressed_data += compress.flush()
    return compressed_data

def genPakoLink(graphMarkdown: str):
    jGraph = {"code": graphMarkdown, "mermaid": {"theme": "standard"}}
    byteStr = json.dumps(jGraph).encode('utf-8')
    deflated = pako_deflate(byteStr)
    dEncode = js_btoa(deflated)
    link_code = dEncode.decode('ascii')
    return link_code

mermaid_link = genPakoLink(mermaid_code)
print("mermaid.live link:")
print('http://mermaid.live/edit#pako:' + mermaid_link)

mermaid.live link:
http://mermaid.live/edit#pako:eNqNWFlzGkkM/itaHjZ2xTaH8UVtpYoAMezGR4FxXvyQnhkBXcy10z12qFT++6qvOWCcWj84VUZSf5I+SR/52fKTAFsDaL3EAv/NMfZxzNk6Y9FLDPSTskxyn6cslrAEJuAlD67PA/rtX/ZhKTBrsJsVhj79XrFruM1YwBOYxRKzFfPx0Olp+LkePuiSeZrTJ8wTh/bPD7PRZO+dPjwn3Ed4zBIfhUgasC3mtVcCPH/JV9hZwSJF9DcwRz9Zx1zyJD50Hi6fpnX3bgeGjzP4B3dwx2K2birH3WR+O6m7dfo2tzvMGn0e5w8j49K7vLmxEOeqQUL+Lr3R1+FyXD6GGuMlDGO5yZKU+zAKWR5gk+NkeF/3W93AiMhBLsjiZpCT53rP+h0Ch68c32ASr3nc8NB0tniqO3kIUy5kkv2mhuOHb/d1L7yAcfIWhwkLYMriIFRuxvE+kQjJK2awPNGegxpJapUx9YDp091XuEVKkxEQeONyY7mk++Qis5DoiD+k7d4jkxvzgfpZnn76RCweQJedwQJD9CW8tJ52KcIYhZ/xVLOqBZJ5da/lbAA9cpqo8YA39ASnDILSCY4kvXpc8ZqRmybWAM7J85EJAcoGuMGrjDAUWM3iXbxeBa+x/z+AyUtNSxbAq/ap4N0DqmeVgJKH5S5Q1WkfrHiILxXrsnPWpeybd0FsjOkj6XyTLGLyLy9rfzoaqr8scB0hcSU1T/B4fVyNrSMSmMV8AH2dcRzYUDJ5d/zrqJRvw+64TZJ1iC4GLQSNyjdwBQjzd3pFdaiKaTE/LctzoQsq8ywGmbFYFdPDwPrUs7AtuCSPMRdpyHaFi+HLCjHwmL89TN+S5spzpGl8SlNnuvMyHjRypzZiNmRtvanLAD6LwU8ijxYBeAmNlGbo8H5sGNNM58+F4UdLLE1pewKoaa6AjojXmlWvnMa43HNqK29

In [9]:
import os
import re
import base64
import tempfile
import gradio as gr
from anthropic import Anthropic
import speech_recognition as sr
from pydub import AudioSegment

# Configuration
SYSTEM_PROMPT = """You are an expert web developer. Create modern, responsive HTML websites using only HTML, CSS, and JavaScript.

REQUIREMENTS:
- Use modern CSS for styling (flexbox, grid, modern selectors)
- Make it fully responsive for mobile devices
- Include proper semantic HTML structure
- Use modern JavaScript ES6+ features when needed
- Create clean, professional designs with good UX
- Include proper accessibility features

IMPORTANT:
- Always respond with complete, working HTML code
- Include all CSS in <style> tags within the HTML
- Include all JavaScript in <script> tags within the HTML
- Make it a single, self-contained HTML file
- Do NOT include any explanations - only return the HTML code
- Do NOT wrap the code in markdown code blocks

The HTML should be ready to run directly in a browser."""

class VoiceProcessor:
    def __init__(self):
        self.recognizer = sr.Recognizer()

    def transcribe_audio(self, audio_file):
        """Convert speech to text using SpeechRecognition"""
        if audio_file is None:
            return "No audio provided"

        try:
            # Convert to wav if needed
            audio = AudioSegment.from_file(audio_file)

            # Export as wav for speech recognition
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp_wav:
                audio.export(tmp_wav.name, format="wav")

                # Transcribe using speech recognition
                with sr.AudioFile(tmp_wav.name) as source:
                    audio_data = self.recognizer.record(source)
                    text = self.recognizer.recognize_google(audio_data)

                # Clean up temp file
                os.unlink(tmp_wav.name)
                return text

        except sr.UnknownValueError:
            return "Could not understand audio - please try speaking more clearly"
        except sr.RequestError as e:
            return f"Speech recognition service error: {e}"
        except Exception as e:
            return f"Error processing audio: {e}"

# Global processor instance
voice_processor = VoiceProcessor()

# Anthropic Client
def get_anthropic_client(api_key):
    """Initialize Anthropic client with provided API key"""
    if not api_key:
        return None
    return Anthropic(api_key=api_key)

def remove_code_block(text):
    """Remove markdown code blocks if present"""
    # Try to match code blocks with language markers
    patterns = [
        r'```(?:html|HTML)\n([\s\S]+?)\n```',  # Match ```html or ```HTML
        r'```\n([\s\S]+?)\n```',               # Match code blocks without language markers
        r'```([\s\S]+?)```'                      # Match code blocks without line breaks
    ]

    for pattern in patterns:
        match = re.search(pattern, text, re.DOTALL)
        if match:
            extracted = match.group(1).strip()
            return extracted

    # If no code block is found, check if the entire text is HTML
    if text.strip().startswith('<!DOCTYPE html>') or text.strip().startswith('<html'):
        return text.strip()

    return text.strip()

def send_to_preview(html_code):
    """Create a safe preview of the HTML code"""
    if not html_code or not html_code.strip():
        return "<div style='padding:2em;color:#666;text-align:center;'>No HTML code to preview</div>"

    # Add a wrapper to ensure proper rendering
    wrapped_code = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
        <meta name="viewport" content="width=device-width, initial-scale=1.0">
        <script>
            // Safe localStorage polyfill for iframe
            const safeStorage = {{
                _data: {{}},
                getItem: function(key) {{ return this._data[key] || null; }},
                setItem: function(key, value) {{ this._data[key] = value; }},
                removeItem: function(key) {{ delete this._data[key]; }},
                clear: function() {{ this._data = {{}}; }}
            }};

            if (!window.localStorage) {{
                Object.defineProperty(window, 'localStorage', {{
                    value: safeStorage,
                    writable: false
                }});
            }}

            window.onerror = function(message, source, lineno, colno, error) {{
                console.error('Error:', message);
            }};
        </script>
    </head>
    <body>
        {html_code}
    </body>
    </html>
    """

    # Encode HTML for safe iframe embedding
    encoded_html = base64.b64encode(wrapped_code.encode('utf-8')).decode('utf-8')
    data_uri = f"data:text/html;charset=utf-8;base64,{encoded_html}"
    iframe = f'<iframe src="{data_uri}" width="100%" height="600px" style="border:1px solid #ddd;border-radius:8px;" sandbox="allow-scripts allow-same-origin allow-forms allow-popups allow-modals"></iframe>'
    return iframe

def transcribe_voice_input(audio_file):
    """Transcribe voice input to text"""
    if audio_file is None:
        return ""

    transcribed_text = voice_processor.transcribe_audio(audio_file)
    return transcribed_text

def generate_html_code(prompt, audio_file, api_key, history, model="claude-3-opus-20240229"):
    """Generate HTML code using Anthropic Claude API with text or voice input"""

    # Handle voice input if provided
    final_prompt = prompt
    if audio_file is not None:
        voice_text = transcribe_voice_input(audio_file)
        if voice_text and not voice_text.startswith("Could not") and not voice_text.startswith("Speech recognition") and not voice_text.startswith("Error"):
            # If we have good voice input, use it (optionally combine with text)
            if prompt.strip():
                final_prompt = f"{prompt.strip()}\n\nAdditional voice input: {voice_text}"
            else:
                final_prompt = voice_text
        elif voice_text.startswith("Could not") or voice_text.startswith("Speech recognition") or voice_text.startswith("Error"):
            if not prompt.strip():
                return f"Voice transcription failed: {voice_text}", history, "", voice_text

    if not final_prompt.strip():
        return "Please enter a description of what you want to build or record voice input.", history, "", ""

    if not api_key:
        return "Please enter your Anthropic API key.", history, "", ""

    try:
        client = get_anthropic_client(api_key)
        if not client:
            return "Failed to initialize Anthropic client.", history, "", ""

        # Prepare messages for the API
        messages = []

        # Add conversation history
        for user_msg, assistant_msg in history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": assistant_msg})

        # Add current user message
        messages.append({"role": "user", "content": final_prompt})

        # Call Anthropic API
        response = client.messages.create(
            model=model,
            max_tokens=4000,
            temperature=0.7,
            system=SYSTEM_PROMPT,
            messages=messages
        )

        # Extract the generated code
        generated_code = response.content[0].text
        clean_code = remove_code_block(generated_code)

        # Update history
        new_history = history + [(final_prompt, clean_code)]

        # Generate preview
        preview_html = send_to_preview(clean_code)

        # Return transcribed text for display
        voice_feedback = ""
        if audio_file is not None:
            voice_feedback = f"Voice input transcribed: '{voice_text}'" if voice_text else "No voice input detected"

        return clean_code, new_history, preview_html, voice_feedback

    except Exception as e:
        error_msg = f"Error generating code: {str(e)}"
        return error_msg, history, "", ""

def clear_all():
    """Clear all inputs and outputs"""
    return "", [], "", "", None

# Available Claude models
CLAUDE_MODELS = [
    {"name": "Claude Opus 4", "id": "claude-3-opus-20240229"},
    {"name": "Claude Sonnet 3.5", "id": "claude-3-5-sonnet-20241022"},
    {"name": "Claude Haiku 3.5", "id": "claude-3-5-haiku-20241022"}
]

# Demo examples
DEMO_EXAMPLES = [
    "Build a modern landing page for a tech startup",
    "Create a responsive portfolio website for a photographer",
    "Design a dashboard for a fitness tracking app",
    "Build a restaurant menu website with modern styling",
    "Create a blog layout with sidebar and article cards",
    "Design a contact form with validation",
    "Build a pricing page with three tier options",
    "Create an image gallery with lightbox effect"
]

# Main Gradio Interface
with gr.Blocks(
    theme=gr.themes.Soft(
        primary_hue="orange",
        secondary_hue="gray",
        font=gr.themes.GoogleFont("Inter")
    ),
    title="Claude HTML Generator with Voice"
) as demo:

    # State for conversation history
    history_state = gr.State([])

    gr.HTML("""
    <div style="text-align: center; margin-bottom: 2rem;">
        <h1 style="color: #333; margin-bottom: 0.5rem;">🤖🎤 Claude HTML Generator</h1>
        <p style="color: #666; font-size: 1.1rem;">Describe your website by typing or speaking - watch Claude bring it to life!</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=1):
            with gr.Group():
                gr.Markdown("### ⚙️ Configuration")
                api_key_input = gr.Textbox(
                    label="Anthropic API Key",
                    placeholder="sk-ant-...",
                    type="password",
                    info="Your Anthropic API key (kept secure in your session)"
                )

                model_dropdown = gr.Dropdown(
                    choices=[model["name"] for model in CLAUDE_MODELS],
                    value="Claude Opus 4",
                    label="Claude Model",
                    info="Choose your preferred Claude model"
                )

            with gr.Group():
                gr.Markdown("### 💬 What do you want to build?")

                with gr.Tabs():
                    with gr.Tab("✍️ Type Description"):
                        prompt_input = gr.Textbox(
                            label="Describe your website",
                            placeholder="e.g., Build a modern landing page for a tech startup with hero section, features, and contact form",
                            lines=3
                        )

                    with gr.Tab("🎤 Voice Description"):
                        audio_input = gr.Audio(
                            label="Record your website description",
                            type="filepath",
                            sources=["microphone"]
                        )
                        voice_feedback = gr.Textbox(
                            label="Voice transcription",
                            placeholder="Your voice input will appear here...",
                            lines=2,
                            interactive=False
                        )

                with gr.Row():
                    generate_btn = gr.Button("🚀 Generate HTML", variant="primary", scale=2)
                    clear_btn = gr.Button("🗑️ Clear", variant="secondary", scale=1)

            with gr.Group():
                gr.Markdown("### 🎯 Quick Examples")
                with gr.Column():
                    example_buttons = []
                    for i, example in enumerate(DEMO_EXAMPLES[:4]):
                        btn = gr.Button(example, variant="secondary", size="sm")
                        example_buttons.append(btn)
                        btn.click(
                            fn=lambda x=example: x,
                            outputs=prompt_input
                        )

        with gr.Column(scale=2):
            with gr.Tabs():
                with gr.Tab("📝 Generated Code"):
                    code_output = gr.Code(
                        label="HTML Code",
                        language="html",
                        lines=20,
                        interactive=True,
                        show_label=False
                    )

                    with gr.Row():
                        download_btn = gr.DownloadButton(
                            "💾 Download HTML",
                            variant="secondary"
                        )

                with gr.Tab("👀 Live Preview"):
                    preview_output = gr.HTML(
                        label="Preview",
                        value="<div style='padding:2em;color:#666;text-align:center;'>Generate some HTML code to see the preview here!</div>"
                    )

                with gr.Tab("💬 Chat History"):
                    chat_output = gr.Chatbot(
                        label="Conversation History",
                        height=400,
                        show_label=False
                    )

    with gr.Row():
        gr.Markdown("""
        ### 🎤 Voice Input Instructions:
        1. **Click the microphone icon** in the "Voice Description" tab
        2. **Speak clearly** about your website requirements
        3. **Click Generate** to create your HTML
        4. **Combine text + voice**: You can use both text input and voice input together!
        """)

    # Event handlers
    def on_generate(prompt, audio_file, api_key, model_name, history):
        # Map model name to model ID
        model_id = next((model["id"] for model in CLAUDE_MODELS if model["name"] == model_name), "claude-3-opus-20240229")

        code, new_history, preview, voice_text = generate_html_code(prompt, audio_file, api_key, history, model_id)

        # Format history for chatbot display
        chat_history = []
        for user_msg, assistant_msg in new_history:
            chat_history.append([user_msg, f"Generated HTML code with {len(assistant_msg)} characters"])

        return code, new_history, preview, chat_history, "", voice_text

    def on_clear():
        return "", [], "", [], "<div style='padding:2em;color:#666;text-align:center;'>Generate some HTML code to see the preview here!</div>", None, ""

    def update_preview(code):
        if code.strip():
            return send_to_preview(code)
        return "<div style='padding:2em;color:#666;text-align:center;'>No HTML code to preview</div>"

    def prepare_download(code):
        if code.strip():
            return gr.DownloadButton(
                "💾 Download HTML",
                value=code,
                filename="generated_website.html",
                variant="secondary"
            )
        return gr.DownloadButton("💾 Download HTML", variant="secondary")

    # Real-time voice transcription for feedback
    def on_audio_change(audio_file):
        if audio_file is not None:
            transcribed = transcribe_voice_input(audio_file)
            return transcribed
        return ""

    # Wire up the events
    generate_btn.click(
        fn=on_generate,
        inputs=[prompt_input, audio_input, api_key_input, model_dropdown, history_state],
        outputs=[code_output, history_state, preview_output, chat_output, prompt_input, voice_feedback]
    )

    clear_btn.click(
        fn=on_clear,
        outputs=[prompt_input, history_state, code_output, chat_output, preview_output, audio_input, voice_feedback]
    )

    # Update voice feedback when audio changes
    audio_input.change(
        fn=on_audio_change,
        inputs=audio_input,
        outputs=voice_feedback
    )

    # Update preview when code changes
    code_output.change(
        fn=update_preview,
        inputs=code_output,
        outputs=preview_output
    )

    # Update download button when code changes
    code_output.change(
        fn=prepare_download,
        inputs=code_output,
        outputs=download_btn
    )

    gr.HTML("""
    <div style="text-align: center; margin-top: 2rem; padding: 1rem; border-top: 1px solid #eee;">
        <p style="color: #666; font-size: 0.9rem;">
            🤖 Powered by Anthropic Claude | 🎤 Voice Recognition | 🎨 Built with Gradio
        </p>
        <p style="color: #888; font-size: 0.8rem;">
            <strong>Installation:</strong> pip install gradio anthropic speech-recognition pydub
        </p>
    </div>
    """)

# Launch the app
if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=7862,
        share=False,
        inbrowser=True
    )

/tmp/ipython-input-2765552770.py:343: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chat_output = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>